# Homework 6 — Project data cleaning

**Project:** Emerging Music Artist (Music Discovery)  
**Student:** Sizhou Wang (confirm spelling before submission)  
**Source:** local `9.listens` from the team's ListenBrainz work. Its exact local version is identified by SHA-256 below; the original download date is not recorded in this folder.  
**Assignment:** `Module_6_Homework.pdf`, pp. 1–3.  

This is an individual, full-dataset audit. The source is unchanged. The notebook runs from the top and writes a full cleaned Parquet plus a 20-row review sample under `outputs/`. The 14-item checklist and five record checks appear below. Team decisions are labeled **HUMAN DECISION**. No MusicBrainz table is merged here.

## 1. Goal, grain, and rules — written before cleaning

**Project vision:** An editorial lead may compare emerging artists for extra playlist exposure (candidate Vision A in `project-start.md`; not a final team decision). **Question:** Which artists show a growing audience with continued engagement? The proposed comparison is distinct listeners per artist per month. This cleaning assignment checks whether listen events and artist identities can support that question; it does not claim an emerging-artist ranking.

**Raw grain:** one ListenBrainz submitted listen. Candidate event key: `(user_id, timestamp, recording_msid)`; verify uniqueness rather than assume it. A repeated `recording_msid` or `user_id` is expected across separate events.

| Rule | Reason | Check |
|---|---|---|
| Preserve `9.listens` and record SHA-256, size, and UTC coverage | Trace source version | source exists; hash and counts |
| Parse Unix seconds to timezone-aware UTC; preserve original integer | Avoid timezone ambiguity | failed parse and min/max |
| Strip only leading/trailing whitespace on names; preserve case, internal spaces, Unicode and IDs | Avoid joining distinct artists or altering identifiers | changed counts and before/after examples |
| Keep missing MusicBrainz IDs as missing, and preserve multi-artist arrays | Name-only inference and first-artist attribution would be unsafe | missingness and multi-artist counts |
| Derive duration in milliseconds from `duration_ms`, otherwise seconds × 1000; flag invalid values | Source uses two units | conversion failure and range checks |
| Keep all events, including historical and unusual durations; flag suspicious ones | Old imports and long tracks can be real | raw = final row count and flag counts |
| Do not deduplicate without confirmed duplicate submissions | Same recording can be played repeatedly | exact row and candidate-key duplicate counts |

**HUMAN DECISION:** The team still needs to approve an emerging-artist definition, observation window, repeat-listen interval, artist attribution for collaborations, and treatment of events without MBIDs. This notebook uses no unsupported artist-name to MBID merge.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import sys
import duckdb
import pandas as pd
from IPython.display import display, Markdown

ROOT = Path.cwd()
SOURCE = ROOT / "9.listens"
OUT = ROOT / "outputs"
OUT.mkdir(exist_ok=True)
CLEAN = OUT / "homework6_clean.parquet"
SAMPLE = OUT / "homework6_sample.csv"
assert SOURCE.is_file(), f"Missing source: {SOURCE}"

con = duckdb.connect()
con.execute("SET TimeZone='UTC'")
con.execute("SET preserve_insertion_order=true")
print("Python", sys.version.split()[0], "| DuckDB", duckdb.__version__, "| Pandas", pd.__version__)
print("Source:", SOURCE.name, "| bytes:", SOURCE.stat().st_size)

Python 3.13.15 | DuckDB 1.5.6 | Pandas 2.3.3
Source: 9.listens | bytes: 3214994924


### 1.1 Source fingerprint and raw preview

The SHA-256 identifies the exact local file used. The bounded preview is for inspection; all audit queries below scan the entire source.

In [2]:
digest = hashlib.sha256()
with SOURCE.open("rb") as file:
    for chunk in iter(lambda: file.read(8 * 1024 * 1024), b""):
        digest.update(chunk)
source_sha256 = digest.hexdigest()
print("9.listens SHA-256:", source_sha256)

RAW_SQL = f"""read_json('{SOURCE.as_posix()}', format='newline_delimited',
    columns={{user_id:'BIGINT', user_name:'VARCHAR', timestamp:'BIGINT',
             track_metadata:'JSON', recording_msid:'VARCHAR'}})"""
con.execute(f"CREATE OR REPLACE VIEW raw AS SELECT * FROM {RAW_SQL}")
print("Raw columns and types:")
display(con.execute("DESCRIBE raw").df())
display(con.execute("""SELECT user_id, user_name, timestamp, recording_msid,
    json_extract_string(track_metadata, '$.artist_name') AS artist_name,
    json_extract_string(track_metadata, '$.track_name') AS track_name
    FROM raw LIMIT 5""").df())

9.listens SHA-256: 54cfe730df6eaf3aac9dd16421a262f96fd69887a6216658c699ec738bb75fcb
Raw columns and types:


,column_name,column_type,null,key,default,extra
0,user_id,BIGINT,YES,None,None,None
1,user_name,VARCHAR,YES,None,None,None
2,timestamp,BIGINT,YES,None,None,None
3,track_metadata,JSON,YES,None,None,None
4,recording_msid,VARCHAR,YES,None,None,None


,user_id,user_name,timestamp,recording_msid,artist_name,track_name
0,39653,galen8183,1790545436,13a11b06-d55c-4eb3-a2d7-4121a15e788f,Radiator Hospital,Are You Feeling Me?
1,93908,snowsuit3207,1790553602,a4167d77-26e9-47c7-9fa9-c7ba5fddc756,League Of Legends • Rob Robinson,Deceit
2,30220,miasinterestinglife,1790553558,5439531a-db20-4aa8-9da4-0a15da0bb6f3,Die drei ???,146 - Der Biss der Bestie - Teil 36
3,137514,raspybxrry,1790553524,1ff57c3c-c771-40e3-9d62-37550202924a,CORTIS,MOTION (feat. Juicy J)
4,171240,dR3aMe1,1790553534,e98fbdba-4812-4445-b26b-a12089ecb91f,.｡*ﾟ+.*.𝓒𝓾𝓻𝓻𝓮𝓷｡+..｡*ﾟ+,worldwidesuperstar


### 1.2 Full-source profile before changes

Missing means SQL `NULL` or blank text after trimming. Artist and recording MBIDs are metadata fields, not required for a submitted listen.

In [3]:
con.execute("""CREATE OR REPLACE VIEW extracted AS
SELECT user_id, user_name, timestamp AS listened_at_unix, recording_msid,
       track_metadata,
       json_extract_string(track_metadata, '$.track_name') AS track_name_raw,
       json_extract_string(track_metadata, '$.artist_name') AS artist_name_raw,
       json_extract_string(track_metadata, '$.release_name') AS release_name_raw,
       json_extract_string(track_metadata, '$.additional_info.recording_mbid') AS recording_mbid_raw,
       json_extract(track_metadata, '$.additional_info.artist_mbids') AS artist_mbids_raw,
       json_extract_string(track_metadata, '$.additional_info.duration_ms') AS duration_ms_raw,
       json_extract_string(track_metadata, '$.additional_info.duration') AS duration_s_raw
FROM raw""")

profile = con.execute("""SELECT COUNT(*) AS raw_rows,
    COUNT(DISTINCT user_id) AS distinct_user_ids,
    COUNT(DISTINCT user_name) AS distinct_user_names,
    COUNT(DISTINCT recording_msid) AS distinct_recording_msids,
    MIN(to_timestamp(listened_at_unix)) AS first_utc,
    MAX(to_timestamp(listened_at_unix)) AS last_utc,
    COUNT(*) FILTER (WHERE user_id IS NULL) AS missing_user_id,
    COUNT(*) FILTER (WHERE user_name IS NULL OR trim(user_name)='') AS missing_user_name,
    COUNT(*) FILTER (WHERE recording_msid IS NULL OR trim(recording_msid)='') AS missing_recording_msid,
    COUNT(*) FILTER (WHERE artist_name_raw IS NULL OR trim(artist_name_raw)='') AS missing_artist_name,
    COUNT(*) FILTER (WHERE track_name_raw IS NULL OR trim(track_name_raw)='') AS missing_track_name,
    COUNT(*) FILTER (WHERE release_name_raw IS NULL OR trim(release_name_raw)='') AS missing_release_name,
    COUNT(*) FILTER (WHERE recording_mbid_raw IS NULL OR trim(recording_mbid_raw)='') AS missing_recording_mbid,
    COUNT(*) FILTER (WHERE artist_mbids_raw IS NULL OR json_array_length(artist_mbids_raw)=0) AS missing_artist_mbids,
    COUNT(*) FILTER (WHERE user_name <> trim(user_name)) AS user_name_trim_changes,
    COUNT(*) FILTER (WHERE artist_name_raw <> trim(artist_name_raw)) AS artist_name_trim_changes,
    COUNT(*) FILTER (WHERE track_name_raw <> trim(track_name_raw)) AS track_name_trim_changes,
    COUNT(*) FILTER (WHERE release_name_raw <> trim(release_name_raw)) AS release_name_trim_changes,
    COUNT(*) FILTER (WHERE listened_at_unix <= 0 OR listened_at_unix > epoch(current_timestamp)) AS invalid_or_future_time,
    COUNT(*) FILTER (WHERE listened_at_unix < epoch(timestamp '2010-01-01')) AS pre_2010_events,
    COUNT(*) FILTER (WHERE duration_ms_raw IS NOT NULL AND trim(duration_ms_raw)<>'' AND try_cast(duration_ms_raw AS DOUBLE) IS NULL) AS failed_ms_conversions,
    COUNT(*) FILTER (WHERE duration_s_raw IS NOT NULL AND trim(duration_s_raw)<>'' AND try_cast(duration_s_raw AS DOUBLE) IS NULL) AS failed_s_conversions,
    COUNT(*) FILTER (WHERE json_array_length(artist_mbids_raw)>1) AS multi_artist_events
FROM extracted""").df().iloc[0].to_dict()
display(pd.DataFrame([profile]).T.rename(columns={0:"value"}))

,value
raw_rows,5022599
distinct_user_ids,24625
distinct_user_names,24625
distinct_recording_msids,1334165
first_utc,2005-02-13 23:55:46+00:00
last_utc,2026-09-28 23:56:23+00:00
missing_user_id,0
missing_user_name,0
missing_recording_msid,0
missing_artist_name,0


### 1.3 Duplicates and text examples

An exact duplicate compares **all five raw columns**, including full metadata. The candidate event key is tested separately. Repeated user IDs and recording MSIDs are legitimate across events.

In [4]:
duplicate_audit = con.execute("""WITH exact_groups AS (
 SELECT user_id,user_name,timestamp,recording_msid,track_metadata,COUNT(*) AS n
 FROM raw GROUP BY ALL HAVING COUNT(*)>1),
 event_groups AS (
 SELECT user_id,timestamp,recording_msid,COUNT(*) AS n
 FROM raw GROUP BY ALL HAVING COUNT(*)>1)
 SELECT (SELECT COUNT(*) FROM exact_groups) AS exact_duplicate_groups,
        (SELECT COALESCE(SUM(n-1),0) FROM exact_groups) AS exact_excess_rows,
        (SELECT COUNT(*) FROM event_groups) AS repeated_event_key_groups,
        (SELECT COALESCE(SUM(n-1),0) FROM event_groups) AS repeated_event_key_excess_rows""").df().iloc[0].to_dict()
display(pd.DataFrame([duplicate_audit]).T.rename(columns={0:"count"}))
print("Repeated user_id events:", profile['raw_rows']-profile['distinct_user_ids'])
print("Repeated recording_msid events:", profile['raw_rows']-profile['distinct_recording_msids'])
display(con.execute("""SELECT user_id,user_name AS before,trim(user_name) AS after
FROM extracted WHERE user_name<>trim(user_name) ORDER BY user_id LIMIT 5""").df())

,count
exact_duplicate_groups,0.0
exact_excess_rows,0.0
repeated_event_key_groups,0.0
repeated_event_key_excess_rows,0.0


Repeated user_id events: 4997974
Repeated recording_msid events: 3688434


,user_id,before,after
0,152466,fandetacu,fandetacu
1,152466,fandetacu,fandetacu
2,152466,fandetacu,fandetacu
3,152466,fandetacu,fandetacu
4,152466,fandetacu,fandetacu


## 2. Clean the full source

The output keeps the original Unix timestamp and username for traceability. A source event is retained even if an enrichment field is missing. The clean file has one row per source event; no merge or imputation occurs.

In [5]:
CLEAN_SQL = """SELECT
  user_id,
  user_name AS user_name_original,
  NULLIF(trim(user_name),'') AS user_name,
  timestamp AS listened_at_unix,
  to_timestamp(timestamp) AS listened_at_utc,
  recording_msid,
  NULLIF(trim(json_extract_string(track_metadata,'$.track_name')),'') AS track_name,
  NULLIF(trim(json_extract_string(track_metadata,'$.artist_name')),'') AS artist_name,
  NULLIF(trim(json_extract_string(track_metadata,'$.release_name')),'') AS release_name,
  NULLIF(trim(json_extract_string(track_metadata,'$.additional_info.recording_mbid')),'') AS recording_mbid,
  CAST(json_extract(track_metadata,'$.additional_info.artist_mbids') AS VARCHAR) AS artist_mbids_json,
  CASE
    WHEN try_cast(json_extract_string(track_metadata,'$.additional_info.duration_ms') AS DOUBLE) IS NOT NULL
      THEN try_cast(json_extract_string(track_metadata,'$.additional_info.duration_ms') AS DOUBLE)
    WHEN try_cast(json_extract_string(track_metadata,'$.additional_info.duration') AS DOUBLE) IS NOT NULL
      THEN try_cast(json_extract_string(track_metadata,'$.additional_info.duration') AS DOUBLE)*1000
    ELSE NULL END AS duration_ms,
  timestamp < epoch(timestamp '2010-01-01') AS flag_pre_2010,
  COALESCE(try_cast(json_extract_string(track_metadata,'$.additional_info.duration_ms') AS DOUBLE),
           try_cast(json_extract_string(track_metadata,'$.additional_info.duration') AS DOUBLE)*1000) <= 0
    OR COALESCE(try_cast(json_extract_string(track_metadata,'$.additional_info.duration_ms') AS DOUBLE),
                try_cast(json_extract_string(track_metadata,'$.additional_info.duration') AS DOUBLE)*1000) > 86400000
    AS flag_duration_out_of_range
FROM raw"""
con.execute(f"COPY ({CLEAN_SQL}) TO '{CLEAN.as_posix()}' (FORMAT PARQUET, COMPRESSION ZSTD)")
con.execute(f"CREATE OR REPLACE VIEW clean AS SELECT * FROM read_parquet('{CLEAN.as_posix()}')")
print("Wrote", CLEAN.relative_to(ROOT), "| bytes", CLEAN.stat().st_size)
display(con.execute("SELECT * FROM clean LIMIT 5").df())

Wrote outputs/homework6_clean.parquet | bytes 162124922


,user_id,user_name_original,user_name,listened_at_unix,listened_at_utc,recording_msid,track_name,artist_name,release_name,recording_mbid,artist_mbids_json,duration_ms,flag_pre_2010,flag_duration_out_of_range
0,39653,galen8183,galen8183,1790545436,2026-09-27 21:43:56+00:00,13a11b06-d55c-4eb3-a2d7-4121a15e788f,Are You Feeling Me?,Radiator Hospital,Something Wild,10d83f54-08bc-4812-a177-ebbdbf26f932,None,112000.0,False,False
1,93908,snowsuit3207,snowsuit3207,1790553602,2026-09-28 00:00:02+00:00,a4167d77-26e9-47c7-9fa9-c7ba5fddc756,Deceit,League Of Legends • Rob Robinson,Sessions: Star Guardian Taliyah,None,"[""0976dbd8-e7fe-4161-9a49-58c57f7a7787""]",186860.0,False,False
2,30220,miasinterestinglife,miasinterestinglife,1790553558,2026-09-27 23:59:18+00:00,5439531a-db20-4aa8-9da4-0a15da0bb6f3,146 - Der Biss der Bestie - Teil 36,Die drei ???,Die drei ??? 146: Der Biss der Bestie,None,None,87085.0,False,False
3,137514,raspybxrry,raspybxrry,1790553524,2026-09-27 23:58:44+00:00,1ff57c3c-c771-40e3-9d62-37550202924a,MOTION (feat. Juicy J),CORTIS,MOTION (feat. Juicy J) - Single,None,None,157361.0,False,False
4,171240,dR3aMe1,dR3aMe1,1790553534,2026-09-27 23:58:54+00:00,e98fbdba-4812-4445-b26b-a12089ecb91f,worldwidesuperstar,.｡*ﾟ+.*.𝓒𝓾𝓻𝓻𝓮𝓷｡+..｡*ﾟ+,Crown of Cuteness,None,None,128000.0,False,False


## 3. Full-dataset validation

Counts below use the full `9.listens` source and full cleaned Parquet. Missing values remain missing. No name-only artist mapping or MusicBrainz merge is attempted.

In [6]:
after = con.execute("""SELECT COUNT(*) AS clean_rows,
 COUNT(DISTINCT user_id) AS distinct_user_ids,
 COUNT(DISTINCT recording_msid) AS distinct_recording_msids,
 COUNT(*) FILTER (WHERE user_id IS NULL) AS missing_user_id,
 COUNT(*) FILTER (WHERE user_name IS NULL) AS missing_user_name,
 COUNT(*) FILTER (WHERE recording_msid IS NULL) AS missing_recording_msid,
 COUNT(*) FILTER (WHERE artist_name IS NULL) AS missing_artist_name,
 COUNT(*) FILTER (WHERE track_name IS NULL) AS missing_track_name,
 COUNT(*) FILTER (WHERE release_name IS NULL) AS missing_release_name,
 COUNT(*) FILTER (WHERE recording_mbid IS NULL) AS missing_recording_mbid,
 COUNT(*) FILTER (WHERE artist_mbids_json IS NULL OR artist_mbids_json='[]') AS missing_artist_mbids,
 COUNT(*) FILTER (WHERE duration_ms IS NULL) AS missing_duration,
 COUNT(*) FILTER (WHERE flag_pre_2010) AS pre_2010_events,
 COUNT(*) FILTER (WHERE flag_duration_out_of_range) AS duration_out_of_range,
 MIN(listened_at_utc) AS first_utc, MAX(listened_at_utc) AS last_utc
FROM clean""").df().iloc[0].to_dict()
missing_map = [('user_id','missing_user_id'),('user_name','missing_user_name'),
 ('recording_msid','missing_recording_msid'),('artist_name','missing_artist_name'),
 ('track_name','missing_track_name'),('release_name','missing_release_name'),
 ('recording_mbid','missing_recording_mbid'),('artist_mbids','missing_artist_mbids')]
missing_table = pd.DataFrame([{'field':name,'before':profile[key],'after':after[key]} for name,key in missing_map])
display(missing_table)
display(pd.DataFrame([{'raw_rows':profile['raw_rows'],'clean_rows':after['clean_rows'],
 'excluded_rows':profile['raw_rows']-after['clean_rows'],
 'pre_2010_flagged':after['pre_2010_events'],
 'duration_flagged':after['duration_out_of_range'],
 'missing_duration':after['missing_duration']}]))
assert profile['raw_rows']==after['clean_rows']
assert duplicate_audit['exact_excess_rows']==0
assert duplicate_audit['repeated_event_key_excess_rows']==0
assert all(profile[k]==after[k] for _,k in missing_map)
assert profile['distinct_user_ids']==after['distinct_user_ids']
assert profile['distinct_recording_msids']==after['distinct_recording_msids']
assert profile['first_utc']==after['first_utc'] and profile['last_utc']==after['last_utc']
print("PASS: source/clean row, missingness, identity and time bounds reconcile")

,field,before,after
0,user_id,0,0
1,user_name,0,0
2,recording_msid,0,0
3,artist_name,0,0
4,track_name,0,0
5,release_name,141030,141030
6,recording_mbid,4844752,4844752
7,artist_mbids,4801159,4801159


,raw_rows,clean_rows,excluded_rows,pre_2010_flagged,duration_flagged,missing_duration
0,5022599,5022599,0,48265,1,2311123


PASS: source/clean row, missingness, identity and time bounds reconcile


### 3.1 Conversion and suspicious values

Durations above 24 hours and listens before 2010 are **flags**, not automatic deletions. Missing duration does not mean zero. No source date is shifted out of UTC.

In [7]:
display(con.execute("""SELECT user_id,listened_at_utc,artist_name,track_name,duration_ms,
 flag_pre_2010,flag_duration_out_of_range
 FROM clean WHERE flag_duration_out_of_range OR flag_pre_2010
 ORDER BY flag_duration_out_of_range DESC, listened_at_utc LIMIT 10""").df())
print("Failed duration_ms conversions:",profile['failed_ms_conversions'])
print("Failed duration seconds conversions:",profile['failed_s_conversions'])
print("Future/nonpositive timestamps:",profile['invalid_or_future_time'])
print("Events with multiple artist MBIDs:",profile['multi_artist_events'])

,user_id,listened_at_utc,artist_name,track_name,duration_ms,flag_pre_2010,flag_duration_out_of_range
0,169190,2026-09-28 22:17:34+00:00,George R. R. Martin,A Storm of Swords,171120068.0,False,True
1,201315,2005-02-13 23:55:46+00:00,Madeleine Peyroux,Don't Wait Too Long,NaN,True,<NA>
2,201315,2005-02-13 23:55:47+00:00,Saybia,I Surrender,NaN,True,<NA>
3,201315,2005-02-13 23:55:48+00:00,Arch Enemy,Shadows and Dust,NaN,True,<NA>
4,201315,2005-02-13 23:55:49+00:00,Bullet for My Valentine,Hand of Blood,NaN,True,<NA>
5,201315,2005-02-13 23:55:50+00:00,In Flames,Cloud Connected,NaN,True,<NA>
6,201315,2005-02-13 23:55:51+00:00,Insomnium,The Moment Of Reckoning,NaN,True,<NA>
7,201315,2005-02-13 23:55:52+00:00,The Agony Scene,Sacrifice,NaN,True,<NA>
8,201315,2005-02-13 23:55:53+00:00,Korpiklaani,Wooden Pints,NaN,True,<NA>
9,201315,2005-02-13 23:55:54+00:00,The Awesome Machine,Tomorrow,NaN,True,<NA>


Failed duration_ms conversions: 0
Failed duration seconds conversions: 0
Future/nonpositive timestamps: 0
Events with multiple artist MBIDs: 16648


### 3.2 Metric denominator check

An illustrative quality gate for Vision A counts **submitted listens** and distinct listeners. It does not claim causation, artist eligibility, or an approved ranking. `artist_name` is used only as source text; same-name artists may collide. Missing artist keys remain in the all-event denominator and are excluded only from artist-level groups, with that count shown explicitly.

In [8]:
metric_check = con.execute("""SELECT COUNT(*) AS all_listens,
 COUNT(*) FILTER (WHERE artist_name IS NOT NULL) AS artist_named_listens,
 COUNT(*) FILTER (WHERE artist_name IS NULL) AS no_artist_name,
 COUNT(*) FILTER (WHERE artist_mbids_json IS NOT NULL AND artist_mbids_json<>'[]') AS events_with_artist_mbid,
 COUNT(DISTINCT user_id) AS distinct_listeners,
 COUNT(*) FILTER (WHERE duration_ms IS NOT NULL) AS duration_denominator,
 AVG(duration_ms) AS avg_duration_ms_of_present_values
 FROM clean""").df()
display(metric_check)
assert int(metric_check.all_listens[0])==int(metric_check.artist_named_listens[0]+metric_check.no_artist_name[0])
assert int(metric_check.events_with_artist_mbid[0])<=int(metric_check.all_listens[0])

,all_listens,artist_named_listens,no_artist_name,events_with_artist_mbid,distinct_listeners,duration_denominator,avg_duration_ms_of_present_values
0,5022599,5022599,0,221440,24625,2711476,217167.30501


## 4. Five original → expected → actual record checks

The cases are selected from source conditions: whitespace username, missing recording MBID, multi-artist metadata, historical event, and unusual duration. Expected values are computed from **raw input and the rules above before reading the cleaned row**. The composite event key identifies the result. This also checks a Codex suggestion: stripping username whitespace is accepted only after comparing original and cleaned values, while changing case or inferring missing MBIDs is rejected.

In [9]:
case_conditions = [
 ('username whitespace',"user_name<>trim(user_name)"),
 ('missing recording MBID',"json_extract_string(track_metadata,'$.additional_info.recording_mbid') IS NULL"),
 ('multiple artist MBIDs',"json_array_length(json_extract(track_metadata,'$.additional_info.artist_mbids'))>1"),
 ('historical timestamp',"timestamp<epoch(timestamp '2010-01-01')"),
 ('duration over 24h',"try_cast(json_extract_string(track_metadata,'$.additional_info.duration_ms') AS DOUBLE)>86400000 OR try_cast(json_extract_string(track_metadata,'$.additional_info.duration') AS DOUBLE)>86400")
]
raw_cases=[]
used_keys=set()
for label,condition in case_conditions:
    candidates=con.execute(f"SELECT * FROM raw WHERE {condition} ORDER BY user_id,timestamp,recording_msid LIMIT 20").fetchall()
    chosen=next((row for row in candidates if (row[0],row[2],row[4]) not in used_keys),None)
    assert chosen is not None, f"No unique example for {label}"
    used_keys.add((chosen[0],chosen[2],chosen[4]))
    raw_cases.append((label,chosen))

checks=[]
for label,(user_id,user_name,timestamp,meta_json,recording_msid) in raw_cases:
    meta=json.loads(meta_json)
    info=meta.get('additional_info') or {}
    raw_ms=info.get('duration_ms')
    raw_s=info.get('duration')
    try: expected_ms=float(raw_ms) if raw_ms is not None else float(raw_s)*1000 if raw_s is not None else None
    except (TypeError,ValueError): expected_ms=None
    expected={
      'user_name':user_name.strip() or None,
      'artist_name':(meta.get('artist_name') or '').strip() or None,
      'recording_mbid':(info.get('recording_mbid') or '').strip() or None,
      'listened_at_utc':datetime.fromtimestamp(timestamp,timezone.utc),
      'duration_ms':expected_ms,
      'flag_pre_2010':timestamp<1262304000,
      'flag_duration_out_of_range':None if expected_ms is None else expected_ms<=0 or expected_ms>86400000,
    }
    result=con.execute("""SELECT user_name,artist_name,recording_mbid,listened_at_utc,
      duration_ms,flag_pre_2010,flag_duration_out_of_range FROM clean
      WHERE user_id=? AND listened_at_unix=? AND recording_msid=?""",
      [user_id,timestamp,recording_msid]).df()
    assert len(result)==1
    actual=result.iloc[0].to_dict()
    matches={key:(pd.isna(actual[key]) and value is None) or actual[key]==value for key,value in expected.items()}
    checks.append({'case':label,'event_key':f'{user_id}|{timestamp}|{recording_msid}',
      'original':{'user_name':user_name,'artist_name':meta.get('artist_name'),
                  'recording_mbid':info.get('recording_mbid'),'timestamp':timestamp,
                  'duration_ms':raw_ms,'duration_s':raw_s},
      'expected':expected,'actual':actual,'match':all(matches.values()),'field_matches':matches})

for check in checks:
    print('\nCASE:',check['case'],'| match:',check['match'],'| key:',check['event_key'])
    fields=['user_name','artist_name','recording_mbid','listened_at_utc',
            'duration_ms','duration_s','flag_pre_2010','flag_duration_out_of_range']
    original=check['original']
    display(pd.DataFrame([{'field':field,
      'original':repr(original.get({'listened_at_utc':'timestamp'}.get(field,field))),
      'expected':repr(check['expected'].get(field,'—')),
      'actual':repr(check['actual'].get(field,'—'))} for field in fields]))
    if not check['match']: print('Field matches:',check['field_matches'])
assert all(check['match'] for check in checks)
print('PASS: 5/5 record checks match expected cleaning rules')


CASE: username whitespace | match: True | key: 152466|1790544543|23ec7e22-6227-4266-aa7c-ce7ba5ec75af


,field,original,expected,actual
0,user_name,'fandetacu ','fandetacu','fandetacu'
1,artist_name,'Jay Scøtt','Jay Scøtt','Jay Scøtt'
2,recording_mbid,None,None,None
3,listened_at_utc,1790544543,"datetime.datetime(2026, 9, 27, 21, 29, 3, tzin...","Timestamp('2026-09-27 21:29:03+0000', tz='UTC')"
4,duration_ms,178447,178447.0,178447.0
5,duration_s,None,'—','—'
6,flag_pre_2010,None,False,False
7,flag_duration_out_of_range,None,False,False



CASE: missing recording MBID | match: True | key: 5|1790582359|e6e9bea9-836e-4414-8b07-835ab62f9c70


,field,original,expected,actual
0,user_name,'Schmias','Schmias','Schmias'
1,artist_name,'Carriego','Carriego','Carriego'
2,recording_mbid,None,None,None
3,listened_at_utc,1790582359,"datetime.datetime(2026, 9, 28, 7, 59, 19, tzin...","Timestamp('2026-09-28 07:59:19+0000', tz='UTC')"
4,duration_ms,None,299000.0,299000.0
5,duration_s,299,'—','—'
6,flag_pre_2010,None,False,False
7,flag_duration_out_of_range,None,False,False



CASE: multiple artist MBIDs | match: True | key: 5|1790595448|1ef34051-e51f-4edc-9b3d-cf451d029f79


,field,original,expected,actual
0,user_name,'Schmias','Schmias','Schmias'
1,artist_name,'Trüby Trio & J-Malik','Trüby Trio & J-Malik','Trüby Trio & J-Malik'
2,recording_mbid,'417fcea9-1653-42bf-a1cc-8dd1bc3b4674','417fcea9-1653-42bf-a1cc-8dd1bc3b4674','417fcea9-1653-42bf-a1cc-8dd1bc3b4674'
3,listened_at_utc,1790595448,"datetime.datetime(2026, 9, 28, 11, 37, 28, tzi...","Timestamp('2026-09-28 11:37:28+0000', tz='UTC')"
4,duration_ms,403120,403120.0,403120.0
5,duration_s,None,'—','—'
6,flag_pre_2010,None,False,False
7,flag_duration_out_of_range,None,False,False



CASE: historical timestamp | match: True | key: 17393|1156723487|5dfd39fe-b814-4e93-b990-af93a5112f91


,field,original,expected,actual
0,user_name,'RevenantOne','RevenantOne','RevenantOne'
1,artist_name,'Korn','Korn','Korn'
2,recording_mbid,None,None,None
3,listened_at_utc,1156723487,"datetime.datetime(2006, 8, 28, 0, 4, 47, tzinf...","Timestamp('2006-08-28 00:04:47+0000', tz='UTC')"
4,duration_ms,None,None,nan
5,duration_s,None,'—','—'
6,flag_pre_2010,None,True,True
7,flag_duration_out_of_range,None,None,None



CASE: duration over 24h | match: True | key: 169190|1790633854|b1906d03-2fb7-4bc5-9f60-640eb77b008a


,field,original,expected,actual
0,user_name,'kehrl','kehrl','kehrl'
1,artist_name,'George R. R. Martin','George R. R. Martin','George R. R. Martin'
2,recording_mbid,None,None,None
3,listened_at_utc,1790633854,"datetime.datetime(2026, 9, 28, 22, 17, 34, tzi...","Timestamp('2026-09-28 22:17:34+0000', tz='UTC')"
4,duration_ms,171120068,171120068.0,171120068.0
5,duration_s,None,'—','—'
6,flag_pre_2010,None,False,False
7,flag_duration_out_of_range,None,True,True


PASS: 5/5 record checks match expected cleaning rules


## 5. Review sample and repeatability

The sample contains the five difficult cases above and 15 deterministic events spread through the source (ordered by the candidate event key). It is for review only; all reported counts are from the full dataset. `homework6_sample.csv` has 20 rows.

In [10]:
case_where=' OR '.join(['(user_id=? AND listened_at_unix=? AND recording_msid=?)' for _ in checks])
params=[]
for _,row in raw_cases: params.extend([row[0],row[2],row[4]])
difficult=con.execute(f"SELECT * FROM clean WHERE {case_where}",params).df()
other=con.execute("""SELECT * FROM clean ORDER BY user_id,listened_at_unix,recording_msid LIMIT 30""").df()
sample=pd.concat([difficult,other],ignore_index=True).drop_duplicates(
    subset=['user_id','listened_at_unix','recording_msid']).head(20)
assert len(sample)==20
sample.to_csv(SAMPLE,index=False)
print('Wrote',SAMPLE.relative_to(ROOT),'| rows',len(sample),'| bytes',SAMPLE.stat().st_size)
display(sample[['user_id','listened_at_utc','artist_name','track_name','recording_mbid',
                'flag_pre_2010','flag_duration_out_of_range']].head(20))

Wrote outputs/homework6_sample.csv | rows 20 | bytes 4372


,user_id,listened_at_utc,artist_name,track_name,recording_mbid,flag_pre_2010,flag_duration_out_of_range
0,5,2026-09-28 07:59:19+00:00,Carriego,Train to Shimonada,None,False,False
1,5,2026-09-28 11:37:28+00:00,Trüby Trio & J-Malik,Bad Luck,417fcea9-1653-42bf-a1cc-8dd1bc3b4674,False,False
2,17393,2006-08-28 00:04:47+00:00,Korn,Hollow Life,None,True,<NA>
3,152466,2026-09-27 21:29:03+00:00,Jay Scøtt,Woodstock,None,False,False
4,169190,2026-09-28 22:17:34+00:00,George R. R. Martin,A Storm of Swords,None,False,True
5,5,2026-09-27 16:16:11+00:00,Fink,Ever Since I Was A Kid It Seemed I Collected S...,c576122c-f4ca-4a07-adcd-a757ccc244f6,False,False
6,5,2026-09-28 05:53:30+00:00,Fink,Ever Since I Was A Kid It Seemed I Collected S...,c576122c-f4ca-4a07-adcd-a757ccc244f6,False,False
7,5,2026-09-28 05:58:51+00:00,Silvera & Blenn,What You Are,5161c435-2360-4acd-a325-0d6d0df81e58,False,False
8,5,2026-09-28 06:03:33+00:00,Ralph Myerz & the Jack Herren Band,A Special Morning,cb6d1d6d-3188-41c9-ba2a-369d46821eb0,False,False
9,5,2026-09-28 06:08:40+00:00,Quantic,Life in the Rain,8e6c8211-6754-420b-b131-7232adfe3d32,False,False


### 5.1 Saved output check

This cell verifies the persisted Parquet and CSV by reopening them. For a clean-room rerun, restart the kernel and run all cells from the top. The same source SHA-256, row count, five record checks, and sample contents should reappear.

In [11]:
sample_reloaded=pd.read_csv(SAMPLE)
saved=duckdb.sql(f"SELECT COUNT(*) n FROM read_parquet('{CLEAN.as_posix()}')").fetchone()[0]
assert saved==profile['raw_rows']
assert len(sample_reloaded)==20
assert sample_reloaded[['user_id','listened_at_unix','recording_msid']].duplicated().sum()==0
print('PASS: saved Parquet rows',saved,'| sample rows',len(sample_reloaded))

PASS: saved Parquet rows 5022599 | sample rows 20


## 6. Assignment checklist and decision log

Statuses and evidence are produced from the executed checks. **Not applicable** is used only for a table merge, because this assignment uses one raw file. The cleaning rules were stated in Section 1 before execution.

In [12]:
checklist = [
 (1,'Done','Candidate Vision A, business question, and proposed distinct-listener metric are in Section 1.'),
 (2,'Done',f"One submitted listen per row; candidate composite key has {duplicate_audit['repeated_event_key_excess_rows']} excess rows (Section 1.3)."),
 (3,'Done',f"Local 9.listens from ListenBrainz work; exact file version SHA-256 {source_sha256}; {SOURCE.stat().st_size} bytes; original unchanged. Download date unrecorded (Section 1.1)."),
 (4,'Done',f"Explicit BIGINT/VARCHAR/JSON load; Unix seconds → UTC; duration unit conversion; failures ms={profile['failed_ms_conversions']}, s={profile['failed_s_conversions']} (Sections 1.2, 2)."),
 (5,'Done',f"Trim outer whitespace, preserve case/IDs; changed usernames={profile['user_name_trim_changes']}, artists={profile['artist_name_trim_changes']}, tracks={profile['track_name_trim_changes']}; before/after in Section 1.3."),
 (6,'Done',f"Exact duplicate excess={duplicate_audit['exact_excess_rows']}; repeated candidate-key excess={duplicate_audit['repeated_event_key_excess_rows']}; user/recording IDs repeat legitimately (Section 1.3)."),
 (7,'Done','Before/after missingness table in Section 3; no missing IDs or values imputed or dropped.'),
 (8,'Done',f"Invalid/future times={profile['invalid_or_future_time']}; pre-2010={after['pre_2010_events']}; out-of-range durations={after['duration_out_of_range']}; flagged, not deleted (Section 3.1)."),
 (9,'Not applicable','No merge was performed. Artist MBIDs remain an array and name-only matching is avoided (Section 2).'),
 (10,'Done',f"Raw={profile['raw_rows']}, clean={after['clean_rows']}, excluded=0; distinct users and recordings and UTC range reconcile (Section 3)."),
 (11,'Done',f"All-event denominator={int(metric_check.all_listens[0])}; artist-named denominator={int(metric_check.artist_named_listens[0])}; duration denominator={int(metric_check.duration_denominator[0])}; NULL groups counted separately (Section 3.2)."),
 (12,'Done','Five original → expected → actual difficult-record checks passed (Section 4).'),
 (13,'Done','This executed notebook ran from original data and reopened its saved Parquet and CSV (Section 5.1). Restart/Run All is the repeat command.'),
 (14,'Done','Team decisions and data coverage limits are documented below.')]
display(pd.DataFrame(checklist,columns=['item','status','evidence']))
decision_log = pd.DataFrame([
 ('UTC timestamp','Unix seconds → UTC; keep original','Unambiguous time comparisons','Raw/clean bounds and invalid/future count reconcile'),
 ('Outer whitespace','Trim names only; preserve case and IDs','Avoid accidental whitespace categories','Before/after examples and change counts'),
 ('Duplicates','Keep all events','No exact or candidate-key duplicate rows found','Full-data duplicate audit'),
 ('Missing IDs','Retain NULL; no name inference','Avoid false MusicBrainz identity','Before/after missingness unchanged'),
 ('Duration','Use ms first, else seconds × 1000; flag >24h or ≤0','Mixed source units; unusual values need review','Failed conversion and suspicious-value counts'),
 ('Old events','Flag pre-2010, retain','Historical imports may be real','Full-data time range and five record checks')],
 columns=['rule','action','reason','check'])
display(decision_log)
display(Markdown(f"""**Remaining limits.** Artist MBIDs are missing for {after['missing_artist_mbids']:,} of {after['clean_rows']:,} listens; recording MBIDs are missing for {after['missing_recording_mbid']:,}. Artist-name grouping can conflate namesakes and collaborations. The dump covers submitted ListenBrainz histories and may not represent a streaming platform. The first observed listen is not necessarily first discovery. Exposure and recommendation impressions are absent. **HUMAN DECISION:** the team must approve eligibility, attribution, observation and follow-up windows before ranking artists or estimating retention."""))

,item,status,evidence
0,1,Done,"Candidate Vision A, business question, and pro..."
1,2,Done,One submitted listen per row; candidate compos...
2,3,Done,Local 9.listens from ListenBrainz work; exact ...
3,4,Done,Explicit BIGINT/VARCHAR/JSON load; Unix second...
4,5,Done,"Trim outer whitespace, preserve case/IDs; chan..."
5,6,Done,Exact duplicate excess=0.0; repeated candidate...
6,7,Done,Before/after missingness table in Section 3; n...
7,8,Done,Invalid/future times=0; pre-2010=48265; out-of...
8,9,Not applicable,No merge was performed. Artist MBIDs remain an...
9,10,Done,"Raw=5022599, clean=5022599, excluded=0; distin..."


,rule,action,reason,check
0,UTC timestamp,Unix seconds → UTC; keep original,Unambiguous time comparisons,Raw/clean bounds and invalid/future count reco...
1,Outer whitespace,Trim names only; preserve case and IDs,Avoid accidental whitespace categories,Before/after examples and change counts
2,Duplicates,Keep all events,No exact or candidate-key duplicate rows found,Full-data duplicate audit
3,Missing IDs,Retain NULL; no name inference,Avoid false MusicBrainz identity,Before/after missingness unchanged
4,Duration,"Use ms first, else seconds × 1000; flag >24h o...",Mixed source units; unusual values need review,Failed conversion and suspicious-value counts
5,Old events,"Flag pre-2010, retain",Historical imports may be real,Full-data time range and five record checks


**Remaining limits.** Artist MBIDs are missing for 4,801,159 of 5,022,599 listens; recording MBIDs are missing for 4,844,752. Artist-name grouping can conflate namesakes and collaborations. The dump covers submitted ListenBrainz histories and may not represent a streaming platform. The first observed listen is not necessarily first discovery. Exposure and recommendation impressions are absent. **HUMAN DECISION:** the team must approve eligibility, attribution, observation and follow-up windows before ranking artists or estimating retention.

## Submission notes

- Run from this directory with the `dso576-project` Python 3.13 kernel (`duckdb`, `pandas`, `nbformat`, `pyarrow`, `ipykernel` are in `pyproject.toml` / `uv.lock`). In VS Code choose **Restart Kernel and Run All**. From a terminal, `uv sync --frozen` followed by `.venv/bin/python -m jupyter nbconvert --execute --to notebook --inplace Homework6.ipynb` works if `nbconvert` is installed; otherwise use VS Code/Jupyter.
- GitHub branch required by the PDF: `siz_datafit` is the current local branch. Branch URL and commit ID must refer to the **submitted commit**; they are not filled in until the work is committed and pushed.
- Gradescope requires an individual PDF containing the checklist, decision log, and five record checks. This notebook contains that evidence and can be exported after reviewing your name and final branch/commit details. Do not share the full raw file if its source terms prohibit redistribution.